# Notebook 00b — MiniFASNetV2 5K hyperparameter screening

Uses frozen Notebook 00 resources only. CelebA Tune-Val selects checkpoints and optimizer LRs. LCC is external-development input only for CSMR lambda/gain. The frozen Test-5K is not evaluated. No CASIA or ONNX work occurs here.

In [ ]:
# CONFIG / explicit Kaggle mounts. Freeze before the first candidate run.
CELEBA_ROOT=("/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/"
             "CelebA_Spoof_/CelebA_Spoof")
LCC_FASD_ROOT="/kaggle/input/datasets/aleksandrpikul222/lcc-fasd/LCC_FASD"
FINAL_RESOURCE_DIR="/kaggle/working/minifasnet_final_resources_v1"
MINIFASNET_SOURCE_PATH="/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py"
MINIFASNET_CHECKPOINT_PATH="/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth"
OUTPUT_ROOT="/kaggle/working/minifasnetv2_5k_hparam_tuning"
TUNE_DATA_SEED=43; TRAIN_SEED=100
BATCH_SIZE=128 # If OOM, restart ALL candidates from clean output with 64.
NUM_WORKERS=4; AMP=True; RESUME=True
MAX_EPOCHS=6; EARLY_STOP_PATIENCE=2; EARLIEST_EARLY_STOP_EPOCH=3
WEIGHT_DECAY=1e-4; LABEL_SMOOTHING=.1; GRAD_CLIP=5.
METRIC_TIE_EPS=1e-12; METRIC_VERIFY_EPS=1e-6
VAL_ACER_MAX_REGRESSION_PP=1.0
CROP_SCALE=2.7; INPUT_SIZE=80
CSMR_WARMUP_EPOCHS=1; CSMR_BAND_CENTERS=[.15,.45,.75]
CSMR_BAND_SIGMA=.10; CSMR_GAIN_RANGE=[.65,.90]; CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32
SCRATCH_LR_CANDIDATES=[1e-4,3e-4,1e-3]
PRETRAINED_LR_CANDIDATES=[{'backbone_lr':5e-5,'head_lr':5e-4},
                          {'backbone_lr':1e-4,'head_lr':1e-3},
                          {'backbone_lr':2e-4,'head_lr':1e-3}]
CSMR_LAMBDA_CANDIDATES=[.10,.20,.40]
CSMR_GAIN_CANDIDATES=[('mild',[.75,.95]),('current',[.65,.90]),('strong',[.55,.85])]
assert len(SCRATCH_LR_CANDIDATES)==len(PRETRAINED_LR_CANDIDATES)==len(CSMR_LAMBDA_CANDIDATES)==len(CSMR_GAIN_CANDIDATES)==3


In [ ]:
import csv, hashlib, importlib.util, json, math, os, random, shutil, time, zipfile, copy, gc
from collections import Counter, OrderedDict
from pathlib import Path, PurePosixPath
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import confusion_matrix, roc_curve
from IPython.display import FileLink, display
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )


def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]


def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb


def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)


def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))


def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }


def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores


def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result


def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes


def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    if not logits_all:
        raise RuntimeError("Common LCC VALID evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Stage 1 LCC")
    if len(logits) != len(y) or len(keys) != len(y):
        raise RuntimeError("Stage 1 LCC logits/labels/keys have inconsistent counts")
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(
        scored, on=["path", "label"], how="left", validate="one_to_one"
    )
    full["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    valid = full["status"].astype(str).eq("VALID")
    if full.loc[valid, "pad_logit_score"].isna().any() or int(valid.sum()) != len(keys):
        raise RuntimeError("Stage 1 inference failed to score the exact common LCC VALID set")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full


def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


def strict_json_write(path,payload):
    Path(path).write_text(json.dumps(payload,indent=2,allow_nan=False)+'\n')

def strict_json_load(path):
    return json.loads(Path(path).read_text(),parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))

def ordered_sha(keys):return sha256_text('\n'.join(map(str,keys)))

def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:
        raise ValueError('Val threshold needs finite 3-class labels/scores')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0)
    nr=int(real.sum());na=len(y)-nr;bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)]
    rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]

def resource_tree_sha(root):
    h=hashlib.sha256()
    for p in sorted(Path(root).rglob('*')):
        if p.is_file():h.update(str(p.relative_to(root)).encode()+b'\0'+sha256_file(p).encode()+b'\n')
    return h.hexdigest()
def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Official CropImage._get_new_box geometry, adapted from xyxy face coordinates.
    src_h,src_w=image.shape[:2]
    x1,y1,x2,y2=map(float,bbox_xyxy)
    box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:
        raise ValueError('Invalid MiniFASNet image/bbox geometry')
    used_scale=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale))
    new_w,new_h=box_w*used_scale,box_h*used_scale
    center_x,center_y=box_w/2+x1,box_h/2+y1
    left,top=center_x-new_w/2,center_y-new_h/2
    right,bottom=center_x+new_w/2,center_y+new_h/2
    if left<0: right-=left;left=0
    if top<0: bottom-=top;top=0
    if right>src_w-1: left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1: top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:
        raise RuntimeError(f'MiniFASNet crop escaped image: {(lx,ty,rx,by)}')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty MiniFASNet crop')
    return cv2.resize(patch,(size,size)),(lx,ty,rx,by),used_scale


def mini_tensor(patch):
    if patch.shape!=(INPUT_SIZE,INPUT_SIZE,3) or patch.dtype!=np.uint8:
        raise ValueError('Expected 80x80 uint8 BGR patch')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)


def source_module(path):
    spec=importlib.util.spec_from_file_location('local_minifasnet_v2',path)
    if spec is None or spec.loader is None:raise RuntimeError('Cannot import local MiniFASNet.py')
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    return module


def new_model(module):
    return module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),
        drop_p=.2,num_classes=3,img_channel=3)


def state_fingerprint(state):
    h=hashlib.sha256()
    for key in sorted(state):
        value=state[key].detach().cpu().contiguous()
        h.update(key.encode());h.update(str(value.dtype).encode());h.update(str(tuple(value.shape)).encode())
        h.update(value.numpy().tobytes())
    return h.hexdigest()


def load_pretrained_canonical(module,checkpoint_path):
    raw=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):
        raise RuntimeError('Expected official OrderedDict with module. prefix')
    stripped=OrderedDict((k[7:],v) for k,v in raw.items())
    if list(stripped['prob.weight'].shape)!=[3,128]:raise RuntimeError('Official prob.weight shape differs from [3,128]')
    if tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official conv6 kernel is not 5x5')
    seed_everything(TRAIN_SEED)
    model=new_model(module)
    model_keys=model.state_dict()
    skipped=['prob.weight']
    shape_mismatch=[k for k,v in stripped.items() if k in model_keys and tuple(v.shape)!=tuple(model_keys[k].shape) and k not in skipped]
    unexpected=[k for k in stripped if k not in model_keys]
    missing_pretrained=[k for k in model_keys if k not in stripped and k not in skipped]
    if shape_mismatch or unexpected or missing_pretrained:raise RuntimeError(f'Unrelated checkpoint mismatch: shape={shape_mismatch[:5]} unexpected={unexpected[:5]} missing={missing_pretrained[:5]}')
    compatible=OrderedDict((k,v) for k,v in stripped.items() if k not in skipped)
    missing,unexpected_load=model.load_state_dict(compatible,strict=False)
    if set(missing)!={'prob.weight'} or unexpected_load:raise RuntimeError(f'Loading audit failed: missing={missing}, unexpected={unexpected_load}')
    model.prob=nn.Linear(128,3,bias=False)  # new task semantics, deterministic seed 100
    if not model.prob.weight.requires_grad or not model.conv1.conv.weight.requires_grad:
        raise RuntimeError('Classifier/backbone must remain trainable')
    audit={'schema':'mini_fasnet_pretrained_loading_audit_v1','checkpoint_file':checkpoint_path.name,
        'checkpoint_sha256':sha256_file(checkpoint_path),'checkpoint_bytes':checkpoint_path.stat().st_size,
        'architecture_source_sha256':sha256_file(MINIFASNET_SOURCE),
        'checkpoint_keys':len(raw),'loaded_pretrained_key_count':len(compatible),
        'missing_keys':list(missing),'unexpected_keys':list(unexpected_load),
        'intentionally_skipped_keys':skipped,'shape_mismatch_keys':shape_mismatch,
        'input_size':[80,80],'crop_scale':CROP_SCALE,'channel_order':'BGR',
        'conv6_kernel':[5,5],'embedding_size':128,'classifier_shape':[3,128],
        'total_params':sum(p.numel() for p in model.parameters()),
        'trainable_params':sum(p.numel() for p in model.parameters() if p.requires_grad)}
    return model,audit
import torch.nn.functional as F
BANDS=("LOW","MID","HIGH")
def normalize_unit(x):
    # MiniFASNet checkpoint expects BGR float [0,1]; no MNV3 mean/std.
    return x


def radial_masks(size=INPUT_SIZE, device='cpu'):
    y = torch.arange(size, device=device, dtype=torch.float32) - size // 2
    yy, xx = torch.meshgrid(y, y, indexing='ij')
    radius = torch.sqrt(xx.square() + yy.square()) / math.sqrt(2 * (size // 2) ** 2)
    centers = torch.as_tensor(CSMR_BAND_CENTERS, dtype=torch.float32, device=device)
    masks = torch.exp(-((radius[None] - centers[:, None, None]) ** 2) /
                      (2 * CSMR_BAND_SIGMA ** 2))
    if masks.shape != (3, size, size) or not torch.isfinite(masks).all() or not ((masks >= 0) & (masks <= 1)).all():
        raise RuntimeError('Invalid CSMR radial masks')
    if len(set(map(float, CSMR_BAND_CENTERS))) != 3:
        raise RuntimeError('CSMR band centers must differ')
    return masks, radius


def spectral_views(x, gains, masks):
    if x.ndim != 4 or x.shape[1:] != (3, INPUT_SIZE, INPUT_SIZE) or not torch.isfinite(x).all():
        raise ValueError('CSMR input must be finite [B,3,80,80]')
    x = x.float()
    gains = gains.to(device=x.device, dtype=torch.float32).reshape(-1)
    if len(gains) != len(x) or not ((gains >= 0) & (gains <= 1)).all():
        raise ValueError('Invalid gains')
    spectrum = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2, -1))
    gain_fields = 1 - (1 - gains[:, None, None, None]) * masks[None]
    if CSMR_PRESERVE_DC:
        gain_fields[:, :, INPUT_SIZE // 2, INPUT_SIZE // 2] = 1.0
    perturbed = spectrum[:, None] * gain_fields[:, :, None]
    raw = torch.fft.ifft2(torch.fft.ifftshift(perturbed, dim=(-2, -1)), norm='ortho').real
    clipped = ((raw < 0) | (raw > 1)).float().mean(dim=(2, 3, 4))
    views = raw.clamp(0, 1)
    if not torch.isfinite(views).all():
        raise RuntimeError('Non-finite CSMR intervention')
    return views, gain_fields, clipped


def aligned_margin(logits, labels):
    d = binary_score(logits.float())
    labels = torch.as_tensor(labels, device=d.device)
    if not torch.isin(labels, torch.tensor([0, 1, 2], device=d.device)).all():
        raise ValueError('Invalid PAD label')
    return torch.where(labels == 0, d, -d)


def harmful_margin_loss(clean_margin, worst_margin):
    drop = clean_margin.detach() - worst_margin
    harmful = drop > 0
    per = F.smooth_l1_loss(worst_margin, clean_margin.detach(), reduction='none')
    return (per * harmful.float()).mean(), harmful, drop


def selection_by_margins(margins):
    if margins.ndim != 2 or margins.shape[1] != 3 or not torch.isfinite(margins).all():
        raise ValueError('Expected finite [B,3] margins')
    return margins.argmin(dim=1)


def select_worst_view(model, base, labels, masks, amp, chunk_size):
    if chunk_size < 1:
        raise ValueError('Selection chunk size must be positive')
    was_training = model.training
    model.eval()
    chosen, indexes, all_gains, all_clips = [], [], [], []
    try:
        with torch.no_grad():
            for start in range(0, len(base), chunk_size):
                x = base[start:start+chunk_size]
                y = labels[start:start+chunk_size]
                gains = torch.empty(len(x), device=x.device).uniform_(*CSMR_GAIN_RANGE)
                views, _, clips = spectral_views(x, gains, masks)
                margins = []
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=amp):
                        logits = model(normalize_unit(views[:, band]))
                    margins.append(aligned_margin(logits, y))
                indexes_chunk = selection_by_margins(torch.stack(margins, dim=1))
                ix = torch.arange(len(x), device=x.device)
                chosen.append(views[ix, indexes_chunk].detach())
                indexes.append(indexes_chunk.detach())
                all_gains.append(gains.detach())
                all_clips.append(clips[ix, indexes_chunk].detach())
    finally:
        model.train(was_training)
    return tuple(torch.cat(a) for a in (chosen, indexes, all_gains, all_clips))


def intervention_sanity(masks, device):
    gen = torch.Generator(device=device).manual_seed(TRAIN_SEED)
    x = torch.rand((2, 3, INPUT_SIZE, INPUT_SIZE), generator=gen, device=device)
    gain = torch.tensor([0.65, 0.90], device=device)
    views, fields, clips = spectral_views(x, gain, masks)
    assert views.shape == (2, 3, 3, INPUT_SIZE, INPUT_SIZE)
    assert torch.all(fields[:, :, INPUT_SIZE//2, INPUT_SIZE//2] == 1)
    unity, _, _ = spectral_views(x, torch.ones(2, device=device), masks)
    assert torch.max(torch.abs(unity[:, 0] - x)).item() < 1e-5
    ft = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2,-1))
    modified = ft[:, None] * fields[:, :, None]
    valid = ft.abs()[:, None] > 1e-5
    phase_error = (torch.angle(modified) - torch.angle(ft[:, None])).abs()
    phase_error = torch.minimum(phase_error, 2*math.pi-phase_error)
    assert phase_error.expand_as(valid.expand_as(modified))[valid.expand_as(modified)].max().item() < 1e-4
    report = {}
    for i, band in enumerate(BANDS):
        diff = (views[:, i] - x).abs()
        report[band] = {'mean_absolute_pixel_difference': float(diff.mean()),
                        'max_absolute_pixel_difference': float(diff.max()),
                        'clipped_pixel_fraction': float(clips[:, i].mean())}
        if report[band]['clipped_pixel_fraction'] > 0.05:
            print('WARNING: clipping exceeds 5% for', band)
    return report
class FinalCelebADataset(Dataset):
    def __init__(self,frame,root,records,train=False):
        self.frame=frame.reset_index(drop=True);self.root=Path(root);self.records=records;self.train=train
        missing=set(self.frame.sample_key.astype(str))-set(records)
        if missing:raise RuntimeError(f'CelebA bbox missing {len(missing)} manifest keys')
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i];key=str(row.sample_key);rec=self.records[key]
        if rec.get('status') not in ('VALID','FALLBACK_DET_FAIL','FALLBACK_SUSPICIOUS') or 'crop_factor' in rec:
            raise RuntimeError('Invalid localization-only bbox: '+key)
        image=cv2.imread(str(self.root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.root/key)
        bbox=bbox_jitter(rec) if self.train else rec['bbox_xyxy']
        patch,_,_=mini_crop_bgr(image,bbox)
        if self.train:patch=apply_e1_mild_augmentation(patch)
        return mini_tensor(patch),int(row.three_class_label),key,int(row.attack_code)

class FinalLCCDataset(Dataset):
    def __init__(self,frame,root,records):
        self.rows=[];self.root=Path(root)
        for row in frame.itertuples(index=False):
            if str(row.status)=='VALID':
                rec=records[str(row.path)]
                if rec.get('status')!='VALID' or 'crop_factor' in rec:raise RuntimeError('LCC localization record invalid')
                self.rows.append((str(row.path),int(row.label),rec))
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        key,label,rec=self.rows[i];image=cv2.imread(str(self.root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.root/key)
        patch,_,_=mini_crop_bgr(image,rec['bbox_xyxy'])
        return mini_tensor(patch),label,key,'VALID'


In [ ]:
root=Path(FINAL_RESOURCE_DIR).expanduser()
if not (root/'config.json').is_file() and (root/'minifasnet_final_resources_v1'/'config.json').is_file():
    root=root/'minifasnet_final_resources_v1'
output=Path(OUTPUT_ROOT);tuning_dir=output/'tuning_resources';selection_dir=output/'selection'
for folder in (tuning_dir,selection_dir):folder.mkdir(parents=True,exist_ok=True)
celeba_root=Path(CELEBA_ROOT);lcc_root=Path(LCC_FASD_ROOT)
MINIFASNET_SOURCE=Path(MINIFASNET_SOURCE_PATH)
MINIFASNET_CHECKPOINT=Path(MINIFASNET_CHECKPOINT_PATH)
required=[root/'config.json',root/'provenance.json',root/'manifests/manifest_fingerprints.json',
    root/'manifests/final_train100k.csv',root/'manifests/final_val15k.csv',
    root/'manifests/final_test5k.csv',root/'manifests/lcc_external_dev.csv',
    root/'bbox/celeba_final_bbox_cache.json',root/'bbox/lcc_bbox_cache.json',
    root/'bbox/train_filter_audit.json',MINIFASNET_SOURCE,MINIFASNET_CHECKPOINT]
missing=[str(p) for p in required if not p.is_file()]
if missing:raise FileNotFoundError('Frozen input missing: '+', '.join(missing))
if not celeba_root.is_dir() or not lcc_root.is_dir():raise FileNotFoundError('Set exact CelebA/LCC roots')
resource_config=strict_json_load(root/'config.json');resource_provenance=strict_json_load(root/'provenance.json')
manifest_fp=strict_json_load(root/'manifests/manifest_fingerprints.json')
if resource_config['input']!={'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'}:
    raise RuntimeError('Frozen MiniFASNet preprocessing changed')
for name in ('final_train100k','final_val15k','final_test5k'):
    p=root/'manifests'/(name+'.csv')
    if sha256_file(p)!=manifest_fp[name]['file_sha256']:raise RuntimeError('Frozen manifest SHA mismatch: '+name)
for path,expected in [(root/'bbox/celeba_final_bbox_cache.json',resource_provenance['celeba_bbox_sha256']),
    (root/'bbox/lcc_bbox_cache.json',resource_provenance['lcc_bbox_sha256']),
    (root/'manifests/lcc_external_dev.csv',resource_provenance['lcc_manifest_sha256']),
    (root/'bbox/train_filter_audit.json',resource_provenance['train_filter_audit_sha256'])]:
    if sha256_file(path)!=expected:raise RuntimeError('Frozen resource SHA mismatch: '+str(path))
# The official Test manifest is checked for bundle integrity above; its rows are not opened.
train_full=pd.read_csv(root/'manifests/final_train100k.csv')
val_full=pd.read_csv(root/'manifests/final_val15k.csv')
assert len(train_full)==100000 and len(val_full)==15000
for name,frame in [('final_train100k',train_full),('final_val15k',val_full)]:
    if ordered_sha(frame.sample_key.astype(str))!=manifest_fp[name]['ordered_sample_key_sha256']:
        raise RuntimeError('Ordered source manifest keys differ: '+name)
assert set(train_full.sample_key).isdisjoint(val_full.sample_key)
assert set(train_full.subject_id.astype(str)).isdisjoint(set(val_full.subject_id.astype(str)))

from sklearn.model_selection import train_test_split
def nested_stratified(frame,n,seed):
    idx=np.arange(len(frame));selected,_=train_test_split(idx,train_size=n,
        stratify=frame.three_class_label.astype(int).to_numpy(),random_state=seed)
    chosen=frame.iloc[selected].reset_index(drop=True).copy()
    chosen['tune_sampling_seed']=seed
    return chosen
tune_train=nested_stratified(train_full,5000,TUNE_DATA_SEED)
tune_val=nested_stratified(val_full,2000,TUNE_DATA_SEED)
assert len(tune_train)==5000 and len(tune_val)==2000
assert set(tune_train.sample_key)<=set(train_full.sample_key)
assert set(tune_val.sample_key)<=set(val_full.sample_key)
assert set(tune_train.subject_id.astype(str)).isdisjoint(set(tune_val.subject_id.astype(str)))
tune_fp={}
for name,frame,source_name in [('tune_train5k',tune_train,'final_train100k'),
                               ('tune_val2k',tune_val,'final_val15k')]:
    if set(frame.three_class_label.astype(int))!={0,1,2}:raise RuntimeError('Tuning class missing')
    path=tuning_dir/(name+'.csv');csv_text=frame.to_csv(index=False)
    if path.exists() and path.read_text()!=csv_text:raise RuntimeError('Previously frozen tuning subset differs: '+name)
    path.write_text(csv_text)
    tune_fp[name]={'sample_count':len(frame),
        'class_counts':{str(k):int(v) for k,v in frame.three_class_label.value_counts().items()},
        'ordered_sample_key_sha256':ordered_sha(frame.sample_key.astype(str)),
        'source_manifest_sha256':manifest_fp[source_name]['file_sha256'],
        'sampling_seed':TUNE_DATA_SEED,'file_sha256':sha256_file(path)}
fp_path=tuning_dir/'tune_manifest_fingerprints.json'
if fp_path.exists() and strict_json_load(fp_path)!=tune_fp:raise RuntimeError('Tuning fingerprint drift')
strict_json_write(fp_path,tune_fp)

bbox_cache=strict_json_load(root/'bbox/celeba_final_bbox_cache.json')['records']
selected_keys=set(tune_train.sample_key.astype(str))|set(tune_val.sample_key.astype(str))
if not selected_keys<=set(bbox_cache):raise RuntimeError('Tuning keys missing from frozen bbox cache')
for key in tune_train.sample_key.astype(str):
    rec=bbox_cache[key]
    if rec.get('status') not in ('VALID','FALLBACK_DET_FAIL'):
        raise RuntimeError('Train hygiene status failed: '+key)
    x1,y1,x2,y2=map(float,rec['bbox_xyxy'])
    if min(x2-x1,y2-y1)<48:raise RuntimeError('Train hygiene face size failed: '+key)
if any('crop_factor' in bbox_cache[k] for k in selected_keys):raise RuntimeError('BBox cache is not localization-only')
for frame in (tune_train,tune_val):
    if not all((celeba_root/k).is_file() for k in frame.sample_key.astype(str)):
        raise FileNotFoundError('Tuning image missing under CELEBA_ROOT')
lcc_frame=pd.read_csv(root/'manifests/lcc_external_dev.csv',keep_default_na=False)
lcc_cache=strict_json_load(root/'bbox/lcc_bbox_cache.json')['records']
if set(lcc_frame.path.astype(str))!=set(lcc_cache):raise RuntimeError('Frozen LCC keys differ')
if set(lcc_frame[lcc_frame.status=='VALID'].label.astype(int))!={0,1}:
    raise RuntimeError('LCC needs Real and Attack')
if not all((lcc_root/k).is_file() for k in lcc_frame.path.astype(str)):
    raise FileNotFoundError('LCC image missing under LCC_FASD_ROOT')
resource_hashes={'bundle_tree_sha256':resource_tree_sha(root),
    'tune_train_manifest_sha256':tune_fp['tune_train5k']['file_sha256'],
    'tune_val_manifest_sha256':tune_fp['tune_val2k']['file_sha256'],
    'celeba_bbox_sha256':sha256_file(root/'bbox/celeba_final_bbox_cache.json'),
    'lcc_manifest_sha256':sha256_file(root/'manifests/lcc_external_dev.csv'),
    'lcc_bbox_sha256':sha256_file(root/'bbox/lcc_bbox_cache.json'),
    'minifasnet_source_sha256':sha256_file(MINIFASNET_SOURCE),
    'official_checkpoint_sha256':sha256_file(MINIFASNET_CHECKPOINT)}
strict_json_write(tuning_dir/'resource_fingerprints.json',resource_hashes)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu');use_amp=bool(AMP and device.type=='cuda')
print('Frozen tuning resources:',tune_fp,'LCC candidates:',len(lcc_frame),'device:',device)


In [ ]:
seed_everything(TRAIN_SEED)
module=source_module(MINIFASNET_SOURCE)
scratch_model=new_model(module)
with torch.no_grad():
    if tuple(scratch_model.eval()(torch.zeros(2,3,80,80)).shape)!=(2,3):raise RuntimeError('MiniFASNet logits contract failed')
scratch_state={k:v.detach().cpu().clone() for k,v in scratch_model.state_dict().items()}
scratch_sha=state_fingerprint(scratch_state)
strict_json_write(tuning_dir/'initialization_fingerprints.json',{'scratch_sha256':scratch_sha,
    'train_seed':TRAIN_SEED,
    'parameter_count':sum(p.numel() for p in scratch_model.parameters())})
masks,_=radial_masks(INPUT_SIZE,device)
sanity=intervention_sanity(masks,device)
assert selection_by_margins(torch.tensor([[2.,-1.,0.]])).item()==1
assert harmful_margin_loss(torch.tensor([0.]),torch.tensor([1.]))[0].item()==0
assert harmful_margin_loss(torch.tensor([1.]),torch.tensor([0.]))[0].item()>0
probe=new_model(module).to(device).train()
before={k:v.clone() for k,v in probe.state_dict().items() if 'running_' in k or 'num_batches_tracked' in k}
select_worst_view(probe,torch.rand(2,3,80,80,device=device),torch.tensor([0,1],device=device),masks,use_amp,2)
assert probe.training and all(torch.equal(v,probe.state_dict()[k]) for k,v in before.items())
del probe,scratch_model
print('Canonical scratch SHA:',scratch_sha,'CSMR sanity:',sanity)


In [ ]:
run_registry={}
def candidate_config(initialization,initial_sha,backbone_lr,classifier_lr,csmr_lambda,gain_range):
    return {'schema':'minifasnetv2_5k_candidate_v1','resource_hashes':resource_hashes,
        'initialization':initialization,'initial_state_sha256':initial_sha,
        'train_seed':TRAIN_SEED,'tune_data_seed':TUNE_DATA_SEED,
        'train_manifest_sha256':tune_fp['tune_train5k']['file_sha256'],
        'val_manifest_sha256':tune_fp['tune_val2k']['file_sha256'],
        'model':{'embedding_size':128,'conv6_kernel':[5,5],'drop_p':.2,'num_classes':3,'img_channel':3},
        'input':{'crop_scale':2.7,'size':[80,80],'color':'BGR','range':'[0,1]'},
        'optimizer':'AdamW','backbone_lr':float(backbone_lr),'classifier_lr':float(classifier_lr),
        'weight_decay':WEIGHT_DECAY,'label_smoothing':LABEL_SMOOTHING,'grad_clip':GRAD_CLIP,
        'scheduler':None,'batch_size':BATCH_SIZE,'amp':use_amp,'max_epochs':MAX_EPOCHS,
        'early_stop_patience':EARLY_STOP_PATIENCE,'earliest_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,
        'augmentation':'same E1 bbox jitter .2, flip .5, brightness/contrast .3',
        'csmr':None if csmr_lambda is None else {'lambda':float(csmr_lambda),'gain_range':list(map(float,gain_range)),
            'band_centers':CSMR_BAND_CENTERS,'band_sigma':CSMR_BAND_SIGMA,
            'warmup_epochs':CSMR_WARMUP_EPOCHS,'preserve_dc':CSMR_PRESERVE_DC}}

def tune_loader(frame,train=False,epoch=0):
    dataset=FinalCelebADataset(frame,celeba_root,bbox_cache,train=train)
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    def worker_init(worker_id):
        value=TRAIN_SEED+epoch*1000+worker_id
        random.seed(value);np.random.seed(value);torch.manual_seed(value)
    return DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=train,drop_last=False,
        num_workers=NUM_WORKERS,pin_memory=device.type=='cuda',generator=gen,worker_init_fn=worker_init)

def run_candidate(stage,candidate_name,initialization,backbone_lr,classifier_lr,csmr_lambda=None,gain_range=None):
    global CSMR_GAIN_RANGE
    folder=output/stage/candidate_name;folder.mkdir(parents=True,exist_ok=True)
    canonical_state=scratch_state if initialization=='scratch' else pretrained_state
    initial_sha=scratch_sha if initialization=='scratch' else pretrained_sha
    if initialization=='pretrained' and csmr_lambda is not None:
        raise RuntimeError('CSMR tuning must use scratch initialization only')
    conf=candidate_config(initialization,initial_sha,backbone_lr,classifier_lr,csmr_lambda,gain_range)
    conf_hash=sha256_text(json.dumps(conf,sort_keys=True,separators=(',',':')))
    conf_saved={**conf,'config_hash':conf_hash}
    config_path=folder/'config.json';metrics_path=folder/'metrics.json';best_path=folder/'best.pth';last_path=folder/'last_state.pth'
    if config_path.exists():
        if strict_json_load(config_path)!=conf_saved:raise RuntimeError('Incompatible candidate directory: '+str(folder))
    else:strict_json_write(config_path,conf_saved)
    if conf_hash in run_registry:
        prior=run_registry[conf_hash]
        strict_json_write(folder/'reuse_reference.json',{'config_hash':conf_hash,'source_dir':prior['candidate_dir']})
        return {**prior,'stage':stage,'candidate_name':candidate_name,'reused_or_new':'REUSED'}
    if metrics_path.exists() or best_path.exists() and not last_path.exists():
        if not metrics_path.exists() or not best_path.exists():raise RuntimeError('Incomplete candidate artifacts: '+str(folder))
        saved=strict_json_load(metrics_path)
        if (saved['config_hash']!=conf_hash or saved['resource_hashes']!=resource_hashes or
            saved['checkpoint_sha256']!=sha256_file(best_path) or
            saved['initialization_sha256']!=initial_sha):
            raise RuntimeError('Completed candidate provenance mismatch: '+str(folder))
        result={**saved,'stage':stage,'candidate_name':candidate_name,'candidate_dir':str(folder),'reused_or_new':'REUSED'}
        run_registry[conf_hash]=result
        return result
    seed_everything(TRAIN_SEED)
    model=new_model(module);model.load_state_dict(canonical_state,strict=True)
    if state_fingerprint(model.state_dict())!=initial_sha:raise RuntimeError('Canonical candidate initialization drift')
    model=model.to(device)
    if initialization=='pretrained':
        backbone=[p for name,p in model.named_parameters() if not name.startswith('prob.')]
        groups=[{'params':backbone,'lr':backbone_lr},
                {'params':list(model.prob.parameters()),'lr':classifier_lr}]
    else:groups=[{'params':model.parameters(),'lr':backbone_lr}]
    optimizer=torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY)
    scheduler=None # Existing final notebooks use no scheduler.
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    criterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    history=[];diagnostics=[];class_diagnostics=[];best_epoch=0
    best_acer=float('inf');best_auc=-float('inf');start_epoch=1;stop_reached=False
    if last_path.exists():
        if not RESUME:raise RuntimeError('Incomplete candidate exists; set RESUME=True or clean its directory')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('config_hash')!=conf_hash or state.get('initialization_sha256')!=initial_sha:
            raise RuntimeError('Candidate resume config/initialization mismatch')
        if 'best_auc' not in state:raise RuntimeError('Old resume lacks best_auc; restart candidate')
        model.load_state_dict(state['model_state']);optimizer.load_state_dict(state['optimizer_state'])
        scaler.load_state_dict(state['scaler_state']);history=state['history']
        diagnostics=state['diagnostics'];class_diagnostics=state['class_diagnostics']
        best_epoch=state['best_epoch'];best_acer=state['best_acer'];best_auc=state['best_auc']
        start_epoch=state['epoch']+1;stop_reached=state['stop_reached']
        random.setstate(state['rng_python']);np.random.set_state(state['rng_numpy'])
        torch.set_rng_state(state['rng_torch'])
        if device.type=='cuda' and state['rng_cuda'] is not None:torch.cuda.set_rng_state_all(state['rng_cuda'])
        if not best_path.is_file():raise RuntimeError('Candidate best checkpoint missing on resume')
    CSMR_GAIN_RANGE=list(gain_range) if gain_range is not None else [.65,.90]
    val_loader=tune_loader(tune_val)
    try:
        for epoch in ([] if stop_reached else range(start_epoch,MAX_EPOCHS+1)):
            started=time.monotonic();model.train();train_loader=tune_loader(tune_train,train=True,epoch=epoch)
            losses=[];cls_losses=[];margin_losses=[];clean_ce_values=[];worst_ce_values=[]
            band_counts=torch.zeros(3,dtype=torch.int64);class_bands=torch.zeros((3,3),dtype=torch.int64)
            class_seen=torch.zeros(3,dtype=torch.int64);class_harm=torch.zeros(3,dtype=torch.int64)
            class_drop=torch.zeros(3,dtype=torch.float64)
            csmr_n=harmful_n=0;drop_sum=clean_margin_sum=worst_margin_sum=gain_sum=clip_sum=0.
            print(f'{stage}/{candidate_name} epoch {epoch}/{MAX_EPOCHS}',flush=True)
            for batch_index,(x,y,_,_) in enumerate(train_loader,1):
                x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                if csmr_lambda is not None and epoch>CSMR_WARMUP_EPOCHS:
                    selected,bands,gains,clips=select_worst_view(model,x,y,masks,use_amp,SELECTION_CHUNK_SIZE)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits=model(torch.cat([x,selected],dim=0));clean_logits,worst_logits=logits.chunk(2)
                        ce_clean=criterion(clean_logits,y);ce_worst=criterion(worst_logits,y)
                        cls_loss=.5*ce_clean+.5*ce_worst
                        clean_margin=aligned_margin(clean_logits,y);worst_margin=aligned_margin(worst_logits,y)
                        margin_loss,harmful,drop=harmful_margin_loss(clean_margin,worst_margin)
                        loss=cls_loss+float(csmr_lambda)*margin_loss
                    n=len(y);csmr_n+=n;harmful_n+=int(harmful.sum().cpu())
                    drop_sum+=float(drop.clamp_min(0).sum().detach().cpu())
                    clean_margin_sum+=float(clean_margin.sum().detach().cpu())
                    worst_margin_sum+=float(worst_margin.sum().detach().cpu())
                    gain_sum+=float(gains.sum().cpu());clip_sum+=float(clips.sum().cpu())
                    clean_ce_values.append(float(ce_clean.detach().cpu()))
                    worst_ce_values.append(float(ce_worst.detach().cpu()))
                    for band in range(3):band_counts[band]+=int((bands==band).sum().cpu())
                    for cls in range(3):
                        mask=y==cls;class_seen[cls]+=int(mask.sum().cpu())
                        class_harm[cls]+=int((harmful&mask).sum().cpu())
                        class_drop[cls]+=float(drop.clamp_min(0)[mask].sum().detach().cpu())
                        for band in range(3):class_bands[cls,band]+=int((mask&(bands==band)).sum().cpu())
                else:
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits=model(x);cls_loss=criterion(logits,y);margin_loss=cls_loss.detach()*0;loss=cls_loss
                    clean_ce_values.append(float(cls_loss.detach().cpu()))
                if not torch.isfinite(loss):raise RuntimeError('Non-finite candidate loss')
                scaler.scale(loss).backward();scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                scaler.step(optimizer);scaler.update()
                losses.append(float(loss.detach().cpu()));cls_losses.append(float(cls_loss.detach().cpu()))
                margin_losses.append(float(margin_loss.detach().cpu()))
                if batch_index==1 or batch_index%max(1,len(train_loader)//8)==0:
                    print('  batch',batch_index,'/',len(train_loader),'mean loss',float(np.mean(losses)),flush=True)
            val_logits,val_y,val_keys,_=eval_spatial(model,val_loader,device,use_amp)
            assert_ordered_keys(val_keys,tune_val.sample_key.astype(str).tolist(),'Tune-Val')
            scores=binary_score(torch.from_numpy(val_logits)).numpy()
            threshold=threshold_from_val(val_y,scores)
            val_metrics,_=metrics_celeba(val_y,val_logits,threshold)
            history.append({'epoch':epoch,'train_loss':float(np.mean(losses)),
                'val_acer':val_metrics['acer'],'val_auc':val_metrics['auc'],
                'val_threshold':threshold,'seconds':time.monotonic()-started})
            if csmr_lambda is not None:
                d={'epoch':epoch,'ce_clean':float(np.mean(clean_ce_values)),
                    'ce_worst':float(np.mean(worst_ce_values)) if worst_ce_values else None,
                    'classification_loss':float(np.mean(cls_losses)),
                    'margin_loss':float(np.mean(margin_losses)),'total_loss':float(np.mean(losses)),
                    'harmful_fraction':harmful_n/csmr_n if csmr_n else None,
                    'mean_harmful_margin_drop':drop_sum/harmful_n if harmful_n else None,
                    'mean_clean_aligned_margin':clean_margin_sum/csmr_n if csmr_n else None,
                    'mean_worst_aligned_margin':worst_margin_sum/csmr_n if csmr_n else None,
                    'mean_sampled_gain':gain_sum/csmr_n if csmr_n else None,
                    'clipping_fraction':clip_sum/csmr_n if csmr_n else None}
                for band in range(3):
                    d['worst_'+BANDS[band].lower()+'_fraction']=int(band_counts[band])/csmr_n if csmr_n else None
                diagnostics.append(d)
                for cls,label in enumerate(('Real','Physical','Digital')):
                    n=int(class_seen[cls]);r={'epoch':epoch,'class':label,'n':n,
                        'harmful_fraction':int(class_harm[cls])/n if n else None,
                        'mean_harmful_margin_drop':float(class_drop[cls])/int(class_harm[cls]) if class_harm[cls] else None}
                    for band in range(3):
                        r['worst_'+BANDS[band].lower()+'_fraction']=int(class_bands[cls,band])/n if n else None
                    class_diagnostics.append(r)
            acer=float(val_metrics['acer']);auc=float(val_metrics['auc'])
            better=acer<best_acer-METRIC_TIE_EPS or (abs(acer-best_acer)<=METRIC_TIE_EPS and auc>best_auc+METRIC_TIE_EPS)
            if better:
                best_acer=acer;best_auc=auc;best_epoch=epoch
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                temp=best_path.with_suffix('.pth.tmp')
                torch.save({'state_dict':best_state,'epoch':epoch,'best_val_acer':acer,
                    'best_val_auc':auc,'config_hash':conf_hash},temp);os.replace(temp,best_path)
            stop_reached=epoch==MAX_EPOCHS or (epoch>=EARLIEST_EARLY_STOP_EPOCH and epoch-best_epoch>=EARLY_STOP_PATIENCE)
            state={'config_hash':conf_hash,'initialization_sha256':initial_sha,'epoch':epoch,
                'model_state':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer_state':optimizer.state_dict(),'scheduler_state':scheduler.state_dict() if scheduler else None,
                'scaler_state':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'history':history,'diagnostics':diagnostics,'class_diagnostics':class_diagnostics,
                'rng_python':random.getstate(),'rng_numpy':np.random.get_state(),
                'rng_torch':torch.get_rng_state(),'rng_cuda':torch.cuda.get_rng_state_all() if device.type=='cuda' else None,
                'stop_reached':stop_reached}
            temp=last_path.with_suffix('.pth.tmp');torch.save(state,temp);os.replace(temp,last_path)
            print('  Tune-Val ACER',acer,'AUC',auc,'best epoch',best_epoch,flush=True)
            if stop_reached:break
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('OOM: restart every candidate with BATCH_SIZE=64 in clean output') from exc
    if not best_path.is_file():raise RuntimeError('No Tune-Val selected checkpoint: '+str(folder))
    checkpoint=torch.load(best_path,map_location='cpu',weights_only=True)
    if checkpoint.get('config_hash')!=conf_hash or checkpoint.get('epoch')!=best_epoch:
        raise RuntimeError('Best checkpoint provenance differs')
    model.load_state_dict(checkpoint['state_dict'])
    val_logits,val_y,val_keys,val_codes=eval_spatial(model,val_loader,device,use_amp)
    assert_ordered_keys(val_keys,tune_val.sample_key.astype(str).tolist(),'Frozen Tune-Val')
    locked=threshold_from_val(val_y,binary_score(torch.from_numpy(val_logits)).numpy())
    val_metrics,_=metrics_celeba(val_y,val_logits,locked)
    if abs(val_metrics['acer']-best_acer)>METRIC_VERIFY_EPS or abs(val_metrics['auc']-best_auc)>METRIC_VERIFY_EPS:
        raise RuntimeError('Best checkpoint does not reproduce saved Tune-Val metrics')
    make_prediction_frame(val_keys,val_y,val_codes,val_logits,locked).to_csv(folder/'val_predictions.csv',index=False)
    pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
    if csmr_lambda is not None:
        pd.DataFrame(diagnostics).to_csv(folder/'csmr_diagnostics.csv',index=False)
        pd.DataFrame(class_diagnostics).to_csv(folder/'csmr_class_diagnostics.csv',index=False)
    # LCC is opened only after the best Tune-Val checkpoint and threshold are frozen.
    lcc_ds=FinalLCCDataset(lcc_frame,lcc_root,lcc_cache)
    lcc_loader=DataLoader(lcc_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
        pin_memory=device.type=='cuda')
    lcc_logits,lcc_y,lcc_keys,lcc_predictions=eval_lcc_spatial(model,lcc_loader,device,use_amp,lcc_frame,locked)
    lcc_scores=binary_score(torch.from_numpy(lcc_logits)).numpy()
    lcc_metrics=metrics_binary(lcc_y,lcc_scores,locked,
        coverage=len(lcc_y)/len(lcc_frame),candidate_n=len(lcc_frame))
    lcc_predictions.to_csv(folder/'lcc_predictions.csv',index=False)
    metrics={'schema':'minifasnetv2_5k_candidate_result_v1','config_hash':conf_hash,
        'resource_hashes':resource_hashes,'initialization_sha256':initial_sha,
        'official_checkpoint_sha256':resource_hashes['official_checkpoint_sha256'] if initialization=='pretrained' else None,
        'best_epoch':best_epoch,'best_val_acer':best_acer,'best_val_auc':best_auc,
        'locked_val_threshold':locked,'threshold_source':'CelebA Tune-Val-2K only',
        'checkpoint_sha256':sha256_file(best_path),'val':val_metrics,'lcc':lcc_metrics}
    strict_json_write(metrics_path,metrics)
    result={**metrics,'stage':stage,'candidate_name':candidate_name,
        'candidate_dir':str(folder),'reused_or_new':'NEW_RUN'}
    run_registry[conf_hash]=result
    model.cpu()
    if device.type=='cuda':torch.cuda.empty_cache()
    print('FROZEN',stage,candidate_name,'Val ACER',best_acer,'Val AUC',best_auc,
          'LCC AUC',lcc_metrics['auc'],'LCC HTER',lcc_metrics['hter'])
    return result


In [ ]:
def best_by_val(results):
    if not results:raise ValueError('No optimizer candidates')
    best=results[0]
    for candidate in results[1:]:
        a=candidate['val'];b=best['val']
        if (a['acer']<b['acer']-METRIC_TIE_EPS or
            (abs(a['acer']-b['acer'])<=METRIC_TIE_EPS and a['auc']>b['auc']+METRIC_TIE_EPS)):
            best=candidate
    return best

def metric_row(stage,result,backbone_lr,classifier_lr,csmr_lambda=None,gain_range=None,
               reference_acer=None,eligible=None,selected=False):
    val=result['val'];lcc=result['lcc']
    return {'stage':stage,'candidate_name':result['candidate_name'],
        'initialization':'scratch' if stage!='B' else 'pretrained',
        'backbone_lr':backbone_lr,'classifier_lr':classifier_lr,
        'csmr_lambda':csmr_lambda,'gain_min':gain_range[0] if gain_range else None,
        'gain_max':gain_range[1] if gain_range else None,'best_epoch':result['best_epoch'],
        'val_apcer':val['apcer'],'val_bpcer':val['bpcer'],'val_acer':val['acer'],
        'val_auc':val['auc'],'val_accuracy_3class':val['accuracy_3class'],
        'lcc_apcer':lcc['apcer'],'lcc_bpcer':lcc['bpcer'],'lcc_hter':lcc['hter'],
        'lcc_auc':lcc['auc'],'lcc_detector_coverage':lcc['detector_coverage'],
        'val_acer_delta_vs_scratch_vanilla':val['acer']-reference_acer if reference_acer is not None else None,
        'eligible':eligible,'selected':selected,'reused_or_new':result['reused_or_new'],
        'initialization_sha256':result['initialization_sha256'],
        'official_checkpoint_sha256':result['official_checkpoint_sha256'],
        'candidate_dir':result['candidate_dir']}

stage_a=[]
for name,lr in zip(('lr_1e-4','lr_3e-4','lr_1e-3'),SCRATCH_LR_CANDIDATES):
    stage_a.append(run_candidate('stage_a_scratch_lr',name,'scratch',lr,lr))
assert len(stage_a)==3 and {r['initialization_sha256'] for r in stage_a}=={scratch_sha}
scratch_reference=best_by_val(stage_a)
selected_scratch_lr=float(strict_json_load(Path(scratch_reference['candidate_dir'])/'config.json')['backbone_lr'])
a_rows=[metric_row('A',r,
    strict_json_load(Path(r['candidate_dir'])/'config.json')['backbone_lr'],
    strict_json_load(Path(r['candidate_dir'])/'config.json')['classifier_lr'],
    reference_acer=scratch_reference['val']['acer'],selected=r['config_hash']==scratch_reference['config_hash']) for r in stage_a]
pd.DataFrame(a_rows).to_csv(selection_dir/'stage_a_scratch_lr_results.csv',index=False)
print('Stage A selected scratch LR from Tune-Val only:',selected_scratch_lr)

# Stage A is complete before the official checkpoint is loaded for Stage B.
pretrained_model,loading_audit=load_pretrained_canonical(module,MINIFASNET_CHECKPOINT)
pretrained_state={k:v.detach().cpu().clone() for k,v in pretrained_model.state_dict().items()}
pretrained_sha=state_fingerprint(pretrained_state)
assert loading_audit['intentionally_skipped_keys']==['prob.weight']
assert loading_audit['checkpoint_sha256']==resource_hashes['official_checkpoint_sha256']
strict_json_write(output/'stage_b_pretrained_loading_audit.json',loading_audit)
init_path=tuning_dir/'initialization_fingerprints.json'
initial_fingerprints=strict_json_load(init_path)
initial_fingerprints['pretrained_sha256']=pretrained_sha
strict_json_write(init_path,initial_fingerprints)
del pretrained_model
stage_b=[]
for i,recipe in enumerate(PRETRAINED_LR_CANDIDATES):
    stage_b.append(run_candidate('stage_b_pretrained_lr',f'candidate_{i}','pretrained',
        recipe['backbone_lr'],recipe['head_lr']))
assert len(stage_b)==3 and {r['initialization_sha256'] for r in stage_b}=={pretrained_sha}
pretrained_winner=best_by_val(stage_b)
pretrained_config=strict_json_load(Path(pretrained_winner['candidate_dir'])/'config.json')
selected_pretrained_backbone_lr=float(pretrained_config['backbone_lr'])
selected_pretrained_head_lr=float(pretrained_config['classifier_lr'])
b_rows=[metric_row('B',r,
    strict_json_load(Path(r['candidate_dir'])/'config.json')['backbone_lr'],
    strict_json_load(Path(r['candidate_dir'])/'config.json')['classifier_lr'],
    reference_acer=scratch_reference['val']['acer'],selected=r['config_hash']==pretrained_winner['config_hash']) for r in stage_b]
pd.DataFrame(b_rows).to_csv(selection_dir/'stage_b_pretrained_lr_results.csv',index=False)
optimizer_selection={'schema':'minifasnetv2_5k_optimizer_selection_v1',
    'selection_policy':'minimum Tune-Val ACER, tie maximum Tune-Val AUC; LCC diagnostic only',
    'scratch_lr':selected_scratch_lr,'scratch_reference_dir':scratch_reference['candidate_dir'],
    'scratch_reference_checkpoint_sha256':scratch_reference['checkpoint_sha256'],
    'pretrained_backbone_lr':selected_pretrained_backbone_lr,
    'pretrained_head_lr':selected_pretrained_head_lr,
    'pretrained_checkpoint_sha256':resource_hashes['official_checkpoint_sha256'],
    'tune_manifest_fingerprints':tune_fp}
selection_path=selection_dir/'optimizer_selection_before_csmr.json'
if selection_path.exists() and strict_json_load(selection_path)!=optimizer_selection:
    raise RuntimeError('Previously frozen optimizer selection differs; stop before CSMR')
strict_json_write(selection_path,optimizer_selection)
print('Optimizer recipes FROZEN before CSMR:',optimizer_selection)


In [ ]:
reference_acer=float(scratch_reference['val']['acer'])
regression_limit=reference_acer+VAL_ACER_MAX_REGRESSION_PP/100.
def eligible_csmr(result):return result['val']['acer']<=regression_limit+METRIC_TIE_EPS

def choose_csmr(results,parameter):
    allowed=[(result,value) for result,value in results if eligible_csmr(result)]
    if not allowed:return None
    if parameter=='lambda':
        return min(allowed,key=lambda item:(-item[0]['lcc']['auc'],item[0]['lcc']['hter'],
            item[0]['val']['acer'],float(item[1])))[0]
    # Larger gains mean less attenuation and win only after all metric ties.
    return min(allowed,key=lambda item:(-item[0]['lcc']['auc'],item[0]['lcc']['hter'],
        item[0]['val']['acer'],-float(item[1][0]),-float(item[1][1])))[0]

stage_c=[]
for name,lam in zip(('lambda_0p10','lambda_0p20','lambda_0p40'),CSMR_LAMBDA_CANDIDATES):
    result=run_candidate('stage_c_lambda',name,'scratch',selected_scratch_lr,
        selected_scratch_lr,csmr_lambda=lam,gain_range=[.65,.90])
    stage_c.append((result,lam))
assert len(stage_c)==3 and {r['initialization_sha256'] for r,_ in stage_c}=={scratch_sha}
lambda_winner=choose_csmr(stage_c,'lambda')
c_rows=[metric_row('C',r,selected_scratch_lr,selected_scratch_lr,lam,[.65,.90],
    reference_acer,eligible_csmr(r),lambda_winner is not None and r['config_hash']==lambda_winner['config_hash'])
    for r,lam in stage_c]
pd.DataFrame(c_rows).to_csv(selection_dir/'stage_c_lambda_results.csv',index=False)
if lambda_winner is None:
    strict_json_write(selection_dir/'no_lambda_passed.json',{'reference_val_acer':reference_acer,
        'max_regression_pp':VAL_ACER_MAX_REGRESSION_PP,'results':c_rows})
    raise RuntimeError('No CSMR lambda passed the 1.0 pp Tune-Val ACER regression guard; no frozen hyperparameters')
selected_lambda=float(strict_json_load(Path(lambda_winner['candidate_dir'])/'config.json')['csmr']['lambda'])
print('Stage C selected lambda:',selected_lambda,'Val guard:',regression_limit)

stage_d=[]
for name,gain in CSMR_GAIN_CANDIDATES:
    result=run_candidate('stage_d_gain',name,'scratch',selected_scratch_lr,
        selected_scratch_lr,csmr_lambda=selected_lambda,gain_range=gain)
    stage_d.append((result,gain))
assert len(stage_d)==3 and {r['initialization_sha256'] for r,_ in stage_d}=={scratch_sha}
assert any(r['reused_or_new']=='REUSED' for r,g in stage_d if g==[.65,.90])
gain_winner=choose_csmr(stage_d,'gain')
d_rows=[metric_row('D',r,selected_scratch_lr,selected_scratch_lr,selected_lambda,gain,
    reference_acer,eligible_csmr(r),gain_winner is not None and r['config_hash']==gain_winner['config_hash'])
    for r,gain in stage_d]
pd.DataFrame(d_rows).to_csv(selection_dir/'stage_d_gain_results.csv',index=False)
if gain_winner is None:
    strict_json_write(selection_dir/'no_gain_passed.json',{'reference_val_acer':reference_acer,
        'max_regression_pp':VAL_ACER_MAX_REGRESSION_PP,'results':d_rows})
    raise RuntimeError('No CSMR gain passed the 1.0 pp Tune-Val ACER regression guard; no frozen hyperparameters')
selected_gain=list(strict_json_load(Path(gain_winner['candidate_dir'])/'config.json')['csmr']['gain_range'])
assert selected_lambda in CSMR_LAMBDA_CANDIDATES and selected_gain in [g for _,g in CSMR_GAIN_CANDIDATES]
print('Stage D selected gain range:',selected_gain)


In [ ]:
summary=pd.DataFrame(a_rows+b_rows+c_rows+d_rows)
assert len(summary)==12 and summary.stage.value_counts().to_dict()=={'A':3,'B':3,'C':3,'D':3}
summary.to_csv(selection_dir/'tuning_summary.csv',index=False)
assert len({r['config_hash'] for r in stage_a+stage_b+[r for r,_ in stage_c]+[r for r,_ in stage_d]})==11
frozen={'schema':'minifasnetv2_final_hparams_v1','selection_stage':'5k_tuning',
    'data':{'train_n':5000,'val_n':2000,'data_seed':TUNE_DATA_SEED,
        'train_manifest_sha256':tune_fp['tune_train5k']['file_sha256'],
        'val_manifest_sha256':tune_fp['tune_val2k']['file_sha256'],
        'source_train_manifest_sha256':manifest_fp['final_train100k']['file_sha256'],
        'source_val_manifest_sha256':manifest_fp['final_val15k']['file_sha256']},
    'scratch':{'optimizer':'AdamW','backbone_lr':selected_scratch_lr,'classifier_lr':selected_scratch_lr,
        'weight_decay':WEIGHT_DECAY,'label_smoothing':LABEL_SMOOTHING,'grad_clip':GRAD_CLIP},
    'pretrained':{'optimizer':'AdamW','backbone_lr':selected_pretrained_backbone_lr,
        'classifier_lr':selected_pretrained_head_lr,'weight_decay':WEIGHT_DECAY,
        'label_smoothing':LABEL_SMOOTHING,'grad_clip':GRAD_CLIP,
        'checkpoint_sha256':resource_hashes['official_checkpoint_sha256']},
    'csmr':{'lambda':selected_lambda,'gain_range':selected_gain,
        'band_centers':CSMR_BAND_CENTERS,'band_sigma':CSMR_BAND_SIGMA,
        'warmup_epochs':CSMR_WARMUP_EPOCHS,'preserve_dc':CSMR_PRESERVE_DC},
    'selection_policy':{'optimizer_selection':'Val ACER, then Val AUC',
        'csmr_source_guard':'Val ACER regression <= 1.0 pp',
        'csmr_primary_external_dev':'LCC AUC','csmr_secondary_external_dev':'LCC HTER',
        'lcc_role':'external-development only; not untouched final cross-domain test'},
    'training_seed':TRAIN_SEED,'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,
    'resource_hashes':resource_hashes,
    'selected_run_checkpoint_sha256':{
        'scratch_vanilla':scratch_reference['checkpoint_sha256'],
        'pretrained_vanilla':pretrained_winner['checkpoint_sha256'],
        'scratch_csmr':gain_winner['checkpoint_sha256']}}
frozen_path=selection_dir/'frozen_hparams.json'
if frozen_path.exists() and strict_json_load(frozen_path)!=frozen:
    raise RuntimeError('Existing frozen hyperparameters differ; do not overwrite without a new protocol')
strict_json_write(frozen_path,frozen)
(selection_dir/'frozen_hparams.sha256').write_text(sha256_file(frozen_path)+'  frozen_hparams.json\n')

def format_stage(rows):
    cols=['candidate_name','backbone_lr','classifier_lr','csmr_lambda','gain_min','gain_max',
          'val_acer','val_auc','lcc_hter','lcc_auc','eligible','selected','reused_or_new']
    return '```\n'+pd.DataFrame(rows)[cols].to_string(index=False)+'\n```'
report=['# MiniFASNetV2 5K hyperparameter screening','',
    '## Objective','Freeze one scratch LR, one pretrained LR pair, and one portable scratch-tuned CSMR lambda/gain recipe. One 5K/2K split and one seed are screening evidence only.','',
    '## Frozen nested resources',f'Tune Train5K / Val2K fingerprints: `{tune_fp}`. Resource hashes: `{resource_hashes}`.','',
    '## Stage A — scratch Vanilla LR',format_stage(a_rows),f'Selected scratch LR: {selected_scratch_lr}. Winner chosen by Tune-Val ACER then AUC; LCC did not choose optimizer LR.','',
    '## Stage B — pretrained Vanilla LR',format_stage(b_rows),
    f'Selected backbone/head LR: {selected_pretrained_backbone_lr} / {selected_pretrained_head_lr}. Released feature weights loaded; only prob.weight skipped and classifier replaced.','',
    '## Scratch Vanilla reference',f'Val ACER {reference_acer:.6f}, Val AUC {scratch_reference["val"]["auc"]:.6f}; LCC AUC {scratch_reference["lcc"]["auc"]:.6f}, HTER {scratch_reference["lcc"]["hter"]:.6f}.','',
    '## Stage C — CSMR lambda',format_stage(c_rows),f'Selected lambda: {selected_lambda}; Tune-Val ACER guard <= {regression_limit:.6f}.','',
    '## Stage D — gain severity',format_stage(d_rows),f'Selected gain range: {selected_gain}. An identical current-gain Stage-C run is reused.','',
    '## Frozen configuration',f'`frozen_hparams.json` SHA256 `{sha256_file(frozen_path)}`. Scratch LR, pretrained LR recipe, CSMR lambda, and gain range are frozen for Notebook 01/02/03. One CSMR recipe is used unchanged for scratch and pretrained.','',
    '## Development-domain role','CelebA Tune-Val controls source checkpoint selection, threshold calibration, optimizer choice, and the CSMR source regression guard. LCC-FASD is an external-development robustness signal for CSMR selection; it is not an untouched final cross-domain test.','',
    '## Evaluation boundary','Frozen Test-5K was never evaluated during this search. CASIA-FASD remains untouched. Later Test/LCC numbers must not cause another hyperparameter search.','',
    '## Next step','Run Notebook 01 as 20K scale confirmation with this frozen configuration, then Notebook 02/03 without tuning.']
(selection_dir/'tuning_report.md').write_text('\n'.join(report)+'\n')
print('FROZEN hyperparameters:',frozen_path,'SHA256:',sha256_file(frozen_path))
print(summary[['stage','candidate_name','val_acer','val_auc','lcc_auc','eligible','selected','reused_or_new']].to_string(index=False))


In [ ]:
ZIP_PATH=Path('/kaggle/working/minifasnetv2_5k_hparam_tuning.zip')
selected_best={Path(scratch_reference['candidate_dir'])/'best.pth',
               Path(pretrained_winner['candidate_dir'])/'best.pth',
               Path(gain_winner['candidate_dir'])/'best.pth'}
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(output.rglob('*')):
        if not path.is_file() or path.name=='last_state.pth':continue
        if path.name=='best.pth' and path not in selected_best:continue
        archive.write(path,arcname=output.name+'/'+str(path.relative_to(output)))
with zipfile.ZipFile(ZIP_PATH) as archive:
    names=set(archive.namelist())
    for path in selected_best:
        if output.name+'/'+str(path.relative_to(output)) not in names:
            raise RuntimeError('Selected checkpoint missing from ZIP: '+str(path))
    if output.name+'/selection/frozen_hparams.json' not in names:
        raise RuntimeError('Frozen hyperparameters missing from ZIP')
print('ZIP:',ZIP_PATH,'size MB:',ZIP_PATH.stat().st_size/1e6)
from IPython.display import FileLink, display
display(FileLink(str(ZIP_PATH)))
